# Week 9 Streamlit Demo Model

Train a small inference artifact for the optional Streamlit app. The app will accept only four user-entered features, so this notebook intentionally trains a separate demo model instead of reusing the full project model.

The saved artifact includes the preprocessing imputer and the model together so inference applies the same transformation used during training.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "crmls_week3_cleaned.csv"
MODEL_DIR = PROJECT_ROOT / "models" / "week9_streamlit"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "week9_streamlit"
MODEL_PATH = MODEL_DIR / "streamlit_demo_price_pipeline.joblib"
METRICS_PATH = OUTPUT_DIR / "streamlit_demo_model_metrics.csv"

TARGET = "ClosePrice"
STREAMLIT_FEATURES = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
]

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

PosixPath('/Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/crmls_week3_cleaned.csv')

## Load modeling data

In [2]:
usecols = [TARGET] + STREAMLIT_FEATURES
df = pd.read_csv(DATA_PATH, usecols=usecols)

# Keep rows with valid target values. Feature missingness is handled by the imputer in the saved pipeline.
df = df[df[TARGET].notna() & np.isfinite(df[TARGET]) & (df[TARGET] > 0)].copy()

X = df[STREAMLIT_FEATURES]
y = df[TARGET]

print(f"Rows available for Streamlit demo model: {len(df):,}")
print("Feature missing values:")
print(X.isna().sum())

Rows available for Streamlit demo model: 333,060
Feature missing values:
LivingArea                364
BedroomsTotal               0
BathroomsTotalInteger      50
LotSizeSquareFeet        6333
dtype: int64


## Train pipeline

`SimpleImputer` is saved inside the same sklearn pipeline as the model. `TransformedTargetRegressor` trains on `log1p(ClosePrice)` and returns predictions back in original dollar units.

In [3]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

# Follow the project convention of deriving outlier bounds from training data only.
price_lower_bound = y_train_raw.quantile(0.005)
price_upper_bound = y_train_raw.quantile(0.995)

train_price_mask = y_train_raw.between(price_lower_bound, price_upper_bound)
test_price_mask = y_test_raw.between(price_lower_bound, price_upper_bound)

X_train = X_train_raw.loc[train_price_mask]
y_train = y_train_raw.loc[train_price_mask]
X_test = X_test_raw.loc[test_price_mask]
y_test = y_test_raw.loc[test_price_mask]

print(f"Training-derived ClosePrice bounds: ${price_lower_bound:,.0f} to ${price_upper_bound:,.0f}")
print(f"Training rows after price filter: {len(X_train):,} of {len(X_train_raw):,}")
print(f"Test rows after price filter: {len(X_test):,} of {len(X_test_raw):,}")

feature_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=300,
                learning_rate=0.08,
                l2_regularization=0.05,
                random_state=42,
            ),
        ),
    ]
)

streamlit_model = TransformedTargetRegressor(
    regressor=feature_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

streamlit_model.fit(X_train, y_train)
pred = np.maximum(streamlit_model.predict(X_test), 0)

Training-derived ClosePrice bounds: $190,000 to $8,500,000
Training rows after price filter: 263,843 of 266,448
Test rows after price filter: 65,949 of 66,612


## Evaluate and save artifact

In [4]:
absolute_percentage_error = np.abs((y_test - pred) / y_test)

metrics = {
    "model_name": "streamlit_demo_hist_gradient_boosting",
    "n_train_before_price_filter": len(X_train_raw),
    "n_test_before_price_filter": len(X_test_raw),
    "n_train": len(X_train),
    "n_test": len(X_test),
    "price_lower_bound": price_lower_bound,
    "price_upper_bound": price_upper_bound,
    "mae": mean_absolute_error(y_test, pred),
    "rmse": mean_squared_error(y_test, pred) ** 0.5,
    "mdape": np.median(absolute_percentage_error),
    "r2": r2_score(y_test, pred),
}

artifact = {
    "model": streamlit_model,
    "features": STREAMLIT_FEATURES,
    "target": TARGET,
    "preprocessing": "SimpleImputer(strategy='median') inside model.regressor_ pipeline",
    "training_data": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "price_filter": {
        "source": "training set only",
        "lower_quantile": 0.005,
        "upper_quantile": 0.995,
        "lower_bound": price_lower_bound,
        "upper_bound": price_upper_bound,
    },
    "metrics": metrics,
}

joblib.dump(artifact, MODEL_PATH)
pd.DataFrame([metrics]).to_csv(METRICS_PATH, index=False)

print(f"Saved model artifact to: {MODEL_PATH}")
print(f"Saved metrics to: {METRICS_PATH}")
pd.DataFrame([metrics])

Saved model artifact to: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/models/week9_streamlit/streamlit_demo_price_pipeline.joblib
Saved metrics to: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/outputs/week9_streamlit/streamlit_demo_model_metrics.csv


,model_name,n_train_before_price_filter,n_test_before_price_filter,n_train,n_test,price_lower_bound,price_upper_bound,mae,rmse,mdape,r2
0,streamlit_demo_hist_gradient_boosting,266448,66612,263843,65949,190000.0,8500000.0,446434.577833,730458.539729,0.300265,0.423343


## Inference smoke test

In [5]:
loaded_artifact = joblib.load(MODEL_PATH)
sample_input = pd.DataFrame(
    [{
        "LivingArea": 1800,
        "BedroomsTotal": 3,
        "BathroomsTotalInteger": 2,
        "LotSizeSquareFeet": 7200,
    }]
)[loaded_artifact["features"]]

sample_prediction = loaded_artifact["model"].predict(sample_input)[0]
print(f"Sample predicted price: ${sample_prediction:,.0f}")

Sample predicted price: $855,820
